# NB-08 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-08**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-08.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This applies the anomaly discussion to the difference between invalid records, bad measurements, and genuine rare events. Detection does not by itself justify deleting or replacing observations.

- [Volume III, Chapter 8 — Anomalies, faults, and cyber attacks](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c8)
- [Volume V, Chapter 8 — From source records to research-ready state](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c8)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-08', require_key=True)
print(setup_status)


# NB-08 — Outlier detection is not data repair
Companion to the observation and anomaly discussions in Volumes III and VI.
Prerequisites: NB-07's representation diagnostics and basic robust statistics.
Learn to distinguish missing/invalid data, a known bad print, and a rare but
genuine model congestion event. Then compare retrospective and past-only
detection, preserve lineage, and measure the damage from indiscriminate removal.

**User-approved bounded deterministic corruption overlay:** retain the original
canonical System E price series and alter exactly three records in a copy.
No random market data is generated. These are model-valid observations, not
verified real-world market records. This is a separate temporal experiment,
not a recreation of the original unsaved randomized graph-anomaly experiment.

In [ ]:
NOTEBOOK_ID='NB-08'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

cfg=inputs['config'];start=cfg['evaluation_start']
def displayed(series):
    # Reconstruct original nonfinite states solely for local inspection/plotting.
    return np.asarray([v if s=='finite' else np.nan if s=='missing' else np.inf for v,s in zip(series['values'],series['observation_states'],strict=True)],float)
x={name:displayed(series) for name,series in inputs['series'].items()}
x.update(inputs['source_context']);x['config']=cfg
print('Local original observations:',inputs['series']['original']['values'][start:start+5])
for label,key in [('overlay','causal'),('clean','clean'),('prefix','prefix')]:
    output=execute('seasonal:'+label,'anomaly_scores',inputs['requests']['seasonal:'+label])
    result=output['result']['anomaly_diagnostics']
    assert result['definition_id']=='diagnose.outlier.hampel.seasonal.v1' and not result['repair_applied']
    x[key]={name:np.asarray(result[name],dtype=object if name=='flags' else float) for name in ('score','center','scale','flags','history_count')}
    if label=='overlay':
        tabulation=output['diagnostics']['retained_evaluation']
        assert not tabulation['anomaly_truth_inferred'] and tabulation['label_origin']=='caller_supplied_control_labels'
        x['confusion']=tabulation['counts']
# Verify the saved censoring control against current flags, without applying a repair.
expected=x['overlay'].copy();expected[x['causal']['flags']=='review']=np.nan
np.testing.assert_array_equal(expected,x['censored'])
metrics={}
for name in ['original','overlay','oracle','censored']:
    parents=['seasonal:overlay'] if name=='censored' else []
    output=execute('global:'+name,'retrospective_mad',inputs['requests']['global:'+name],parents)
    result=output['result']['anomaly_diagnostics'];evaluation=output['diagnostics']['retained_evaluation']
    assert result['definition_id']=='diagnose.outlier.mad.retrospective_global.v1' and not result['causal']
    assert result['reference_end_index_exclusive']==8760 and result['reference_policy']=='entire_submitted_panel_retrospective'
    assert evaluation['start_index']==start and evaluation['end_index_exclusive']==8760 and not evaluation['excluded_values_estimated']
    metrics[name]={key:evaluation[key] for key in ('finite','excluded','mean','maximum','absolute_p95')}
    if name=='overlay':x['global_score']=displayed(dict(values=result['score'],observation_states=result['score_states']))
print(json.dumps(cfg,indent=2));print('Overlay faults:',cfg['fault_hours']);print('Immutable input checks: verified manifest and source support')

## Three known faults, one unchanged market path
At hours 6100,6200,6300 insert respectively NaN (missing), infinity (invalid),
and 20 times the maximum absolute pre-evaluation price (known bad print).
The injection size is a teaching control, not a learned market-error rule.
Original values remain immutable and separately accessible. All evaluation
starts at hour 6000, after a fixed warm-up. The threshold is declared, not
optimized against the known fault labels or the held-out extreme.

The strongest retained absolute price after the cutoff is a **model-valid
congestion event**. Nonzero congestion components and line shadow prices
supply physical context. Magnitude alone cannot tell us it is erroneous.

In [ ]:
event=x['event_hour'];print('Retained event hour',event,'price',x['original'][event],'congestion component',x['event_congestion'],'line dual norm',x['event_dual_norm'])
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(np.arange(start,len(x['original'])),x['original'][start:],color='#88C0D0',linewidth=.8);axes[0].scatter([event],[x['original'][event]],color='#EBCB8B',label='Retained valid extreme');axes[0].set(xlabel='Retained hour',ylabel='LMP ($/MWh)',title='Original evaluation path');axes[0].legend(fontsize=8)
t=cfg['fault_hours']['bad_print'];hours=np.arange(t-24,t+25)
axes[1].plot(hours,x['overlay'][hours],color='#BF616A',label='Fault overlay');axes[1].plot(hours,x['original'][hours],color='#88C0D0',label='Original');axes[1].set(xlabel='Retained hour',ylabel='LMP ($/MWh)',title='One known bad print in a separate copy');axes[1].legend(fontsize=8);fig.tight_layout();plt.show()

## Detect using only earlier observations
For hour t, select the same hour of day in the preceding 28 days, excluding t.
Discard nonfinite reference values, require at least 14 observations, and
calculate |x−median|/(1.4826 MAD). Scores above 6 mean **review**, not error.
Insufficient history and zero MAD are explicit unresolved states: neither
becomes an ordinary observation or an infinite z-score. Missing and invalid
values are classified before scoring. The retained hourly index has no DST
ambiguity; a real feed would need UTC timestamps and an explicit calendar.

The comparator uses one median/MAD from the entire overlaid series. It sees
future observations and ignores seasonality, so it is retrospective only.
Neither detector establishes truth. Earlier bad prints can contaminate later
references even in a causal detector; robustness is not perfect immunity.

In [ ]:
print('Truth versus causal status (evaluation only):',json.dumps(x['confusion'],indent=2))
print('Retained extreme causal status:',x['causal']['flags'][event])
valid=np.isfinite(x['global_score'][start:]);causal=x['causal']['score'][start:]
print('Global retrospective reviews:',int(np.sum(x['global_score'][start:][valid]>cfg['threshold'])))
fig,axes=plt.subplots(1,2,figsize=(12,4));hours=np.arange(start,len(x['original']))
for ax,score,title in [(axes[0],x['global_score'][start:],'Retrospective global score'),(axes[1],causal,'Past-only same-hour score')]:
    ok=np.isfinite(score);ax.scatter(hours[ok],score[ok],s=7,color='#88C0D0');ax.axhline(6,color='#BF616A');ax.set_yscale('symlog',linthresh=1);ax.set(xlabel='Retained hour',ylabel='Absolute robust score',title=title)
for ax in axes:ax.set_ylim(0,1.5*max(np.nanmax(causal),np.max(x['global_score'][start:][valid])))
fig.tight_layout();plt.show()
cut=6400;prefix=x['prefix'];np.testing.assert_array_equal(prefix['flags'],x['causal']['flags'][:cut]);np.testing.assert_allclose(prefix['score'],x['causal']['score'][:cut],equal_nan=True)
print('Prefix invariance passed: later observations cannot change earlier results.')

## Version treatment separately from detection
Four named versions remain accessible: original, corruption overlay, known-fault
oracle repair, and blind censoring of every review flag. Oracle repair restores
only the three known injected faults using retained truth. That is possible
solely because this is a controlled experiment; it is not a repair algorithm
available for real unknown errors. Blind censoring is a negative control.

The detector flags 30 model-valid evaluation observations as well as the bad
print, including the genuine congestion extreme. Blind censoring deletes that
extreme and leaves missing/invalid faults unresolved. A real workflow needs
source verification, revision provenance and an explicit treatment decision.
Counts below expose nonfinite exclusions; finite-only means and upper tails
describe available values and do not estimate the missing ones.

In [ ]:
print(json.dumps(metrics,indent=2))
np.testing.assert_array_equal(x['oracle'],x['original']);assert np.isnan(x['censored'][event]);assert np.isfinite(x['oracle'][event])
fig,axes=plt.subplots(1,2,figsize=(11,4));names=['original','oracle','censored']
for ax,key,title in [(axes[0],'maximum','Retained upper extreme'),(axes[1],'absolute_p95','Absolute 95th percentile')]:
    ax.bar(names,[metrics[n][key] for n in names],color=['#88C0D0','#A3BE8C','#BF616A']);ax.set(ylabel='$/MWh',title=title)
fig.tight_layout();plt.show()

## Exercise: should the congestion extreme be removed?
Look at the flag, the original observation and the congestion evidence.
The answer is no: this experiment provides no error evidence for that hour.
Review status is an invitation to investigate. Zero-scale histories likewise
need a separate policy, not silent acceptance. Optional further work would
compare graph residuals on these same retained observations; this notebook
makes no claim that temporal outliers identify every structural anomaly.

In [ ]:
print('Event preserved by known-fault-only treatment:',x['oracle'][event]==x['original'][event])
print('Zero-scale evaluation hours:',int(np.sum(x['causal']['flags'][start:]=='zero_scale')))

In [ ]:
bad=copy.deepcopy(inputs['requests']['global:overlay']);bad['inputs']['detector_settings']['training_end_index_exclusive']=6000;reject('retrospective_mad',bad)
bad=copy.deepcopy(inputs['requests']['seasonal:overlay']);bad['parameters']['retained_evaluation']['known_control_labels'].pop();reject('anomaly_scores',bad)
checked=dict(approved_overlay=cfg,confusion=x['confusion'],metrics=metrics,retained_event_hour=event,new_random_market_data=False,scientific_acceptance=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(checked)
print(summary)